In [10]:
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from liffile import LifFile

In [11]:
def sanitize(name):
    """Replace spaces with underscores, drop periods, and flatten path separators."""
    return (str(name).replace(" ", "_").replace(".", "")
            .replace("/", "_").replace("\\", "_"))


def to_zcyx(data, dims):
    """Reshape an array with arbitrary `dims` (e.g. 'TZCYX', 'ZYX') to (Z, C, Y, X).
    A T axis (if present) is reduced to its first timepoint; missing Z/C become size 1.
    """
    dims = list(dims)
    if "T" in dims:
        ax = dims.index("T")
        data = np.take(data, 0, axis=ax)
        dims.pop(ax)
    for needed in ("Z", "C"):
        if needed not in dims:
            data = np.expand_dims(data, 0)
            dims = [needed] + dims
    order = [dims.index(a) for a in ("Z", "C", "Y", "X")]
    return np.transpose(data, order)


def lif_pixel_sizes(img):
    """Return (z_um, y_um, x_um) from a liffile image; any value is None if unavailable."""
    z_um = y_um = x_um = None
    try:
        coords = img.asxarray().coords
        if "X" in coords and coords["X"].size >= 2:
            x_um = abs(float(coords["X"][1] - coords["X"][0])) * 1e6
        if "Y" in coords and coords["Y"].size >= 2:
            y_um = abs(float(coords["Y"][1] - coords["Y"][0])) * 1e6
        if "Z" in coords and coords["Z"].size >= 2:
            z_um = abs(float(coords["Z"][1] - coords["Z"][0])) * 1e6
    except Exception as e:
        print(f"  [WARN] could not read LIF pixel sizes: {e}")
    return z_um, y_um, x_um


def tif_pixel_sizes(tif):
    """Return (z_um, y_um, x_um) from a tifffile TiffFile; any value is None if unavailable."""
    z_um = y_um = x_um = None
    tags = {t.name: t.value for t in tif.pages[0].tags.values()}
    xr = tags.get("XResolution")
    yr = tags.get("YResolution")
    if xr and xr[0]:
        x_um = xr[1] / xr[0]  # XResolution = pixels per unit -> um per pixel
    if yr and yr[0]:
        y_um = yr[1] / yr[0]
    ij = tif.imagej_metadata or {}
    if "spacing" in ij:
        z_um = float(ij["spacing"])
    return z_um, y_um, x_um


def save_outputs(data, z_um, y_um, x_um, name, output_folder):
    """Write one TIF (pixel size in metadata) and one PNG (middle-z slice of each
    channel) for a (Z, C, Y, X) image. If X or Y pixel size is missing, the output
    name is tagged with INCORRECT_PIXEL_SIZE and no resolution is written.
    """
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    if x_um is None or y_um is None:
        name = f"{name}_INCORRECT_PIXEL_SIZE"
        resolution = None
        metadata = {"axes": "ZCYX"}
        print(f"  [WARN] missing pixel size -> {name}")
    else:
        resolution = (1.0 / x_um, 1.0 / y_um)  # pixels per micron (X, Y)
        metadata = {"axes": "ZCYX", "unit": "micron"}
        if z_um:
            metadata["spacing"] = z_um

    tif_path = output_folder / f"{name}.tif"
    png_path = output_folder / f"{name}.png"
    tifffile.imwrite(tif_path, data, imagej=True, resolution=resolution, metadata=metadata)

    n_channels = data.shape[1]
    mid_z = data.shape[0] // 2
    fig, axes = plt.subplots(1, n_channels, figsize=(5 * n_channels, 5), squeeze=False)
    for c in range(n_channels):
        axes[0, c].imshow(data[mid_z, c], cmap="gray")
        axes[0, c].set_title(f"Channel {c}")
        axes[0, c].axis("off")
    fig.suptitle(f"{name}  (middle z = {mid_z})")
    fig.tight_layout()
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {tif_path.name}  (Z={data.shape[0]}, C={n_channels})")


def process_mixed_folder(input_dir, output_dir):
    """Process a folder recursively:
    - Convert each LIF sub-image to TIF+PNG
    - Re-save each TIF/TIFF to TIF+PNG
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for lif_path in sorted(input_dir.rglob("*.lif")):
        print(f"LIF: {lif_path}")
        with LifFile(lif_path) as lif:
            for img in lif.images:
                sub_name = "".join(img.path)
                try:
                    data = to_zcyx(img.asarray(), tuple(img.dims))
                except Exception as e:
                    print(f"  SKIP {sub_name}: {type(e).__name__}: {e}")
                    continue
                z_um, y_um, x_um = lif_pixel_sizes(img)
                if y_um is None and x_um is not None:
                    y_um = x_um
                name = sanitize(f"{lif_path.stem} {sub_name}")
                save_outputs(data, z_um, y_um, x_um, name, output_dir)

    tif_paths = sorted(list(input_dir.rglob("*.tif")) + list(input_dir.rglob("*.tiff")))
    for tif_path in tif_paths:
        print(f"TIF: {tif_path}")
        with tifffile.TiffFile(tif_path) as tif:
            series = tif.series[0]
            data = to_zcyx(series.asarray(), tuple(series.axes))
            z_um, y_um, x_um = tif_pixel_sizes(tif)
        if y_um is None and x_um is not None:
            y_um = x_um
        name = sanitize(tif_path.stem)
        save_outputs(data, z_um, y_um, x_um, name, output_dir)

    print("mixed-folder conversion done")


In [12]:
def swap_channels_in_folder(input_dir, output_dir):
    """Find all TIFs in `input_dir`, swap channel 0 <-> channel 1, and save the
    re-ordered TIF (pixel size metadata embedded) + a per-channel PNG into `output_dir`.
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    tif_paths = sorted(list(input_dir.rglob("*.tif")) + list(input_dir.rglob("*.tiff")))
    for tif_path in tif_paths:
        print(f"SWAP: {tif_path}")
        with tifffile.TiffFile(tif_path) as tif:
            series = tif.series[0]
            data = to_zcyx(series.asarray(), tuple(series.axes))  # (Z, C, Y, X)
            z_um, y_um, x_um = tif_pixel_sizes(tif)
        if y_um is None and x_um is not None:
            y_um = x_um

        if data.shape[1] < 2:
            print(f"  SKIP - only {data.shape[1]} channel(s), nothing to swap")
            continue

        swapped = data.copy()
        swapped[:, [0, 1]] = data[:, [1, 0]]  # channel 0 <-> channel 1
        name = sanitize(tif_path.stem)
        save_outputs(swapped, z_um, y_um, x_um, name, output_dir)

    print("channel swap done")


In [13]:
def three_to_two_channel_folder(input_dir, output_dir):
    """Find all TIFs in `input_dir`, build a 2-channel TIF where new channel 0 is
    the old channel 2 and new channel 1 is the old channel 0, and save the TIF
    (pixel size metadata preserved) + a per-channel PNG into `output_dir`.
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    tif_paths = sorted(list(input_dir.rglob("*.tif")) + list(input_dir.rglob("*.tiff")))
    for tif_path in tif_paths:
        print(f"3->2 CH: {tif_path}")
        with tifffile.TiffFile(tif_path) as tif:
            series = tif.series[0]
            data = to_zcyx(series.asarray(), tuple(series.axes))  # (Z, C, Y, X)
            z_um, y_um, x_um = tif_pixel_sizes(tif)
        if y_um is None and x_um is not None:
            y_um = x_um

        if data.shape[1] < 3:
            print(f"  SKIP - only {data.shape[1]} channel(s), expected at least 3")
            continue

        # new channel 0 = old channel 2, new channel 1 = old channel 0
        new_data = np.stack([data[:, 2], data[:, 0]], axis=1)  # (Z, 2, Y, X)
        name = sanitize(tif_path.stem)
        save_outputs(new_data, z_um, y_um, x_um, name, output_dir)

    print("3-channel -> 2-channel done")


In [14]:
def correct_4_channel_folder(input_dir, output_dir):
    """Find all TIFs in `input_dir` and resave each as a 2-channel TIF (pixel size
    metadata preserved + per-channel PNG) into `output_dir`. The channel selection
    depends on the original channel count:
      - 4 channels -> (old C2, old C1)  => new C0=old C2, new C1=old C1
      - 5 channels -> (old C1, old C3)  => new C0=old C1, new C1=old C3
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    channel_map = {4: (2, 1), 5: (1, 3)}

    tif_paths = sorted(list(input_dir.rglob("*.tif")) + list(input_dir.rglob("*.tiff")))
    for tif_path in tif_paths:
        print(f"CORRECT: {tif_path}")
        with tifffile.TiffFile(tif_path) as tif:
            series = tif.series[0]
            data = to_zcyx(series.asarray(), tuple(series.axes))  # (Z, C, Y, X)
            z_um, y_um, x_um = tif_pixel_sizes(tif)
        if y_um is None and x_um is not None:
            y_um = x_um

        n_channels = data.shape[1]
        if n_channels not in channel_map:
            print(f"  SKIP - {n_channels} channel(s), expected 4 or 5")
            continue

        c0, c1 = channel_map[n_channels]
        new_data = np.stack([data[:, c0], data[:, c1]], axis=1)  # (Z, 2, Y, X)
        name = sanitize(tif_path.stem)
        save_outputs(new_data, z_um, y_um, x_um, name, output_dir)

    print("4-channel correction done")


In [ ]:
process_mixed_folder(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\Agus_María", r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\FLUORESCENT_CELLS\tifs\agus_maria")

swap_channels_in_folder(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\FLUORESCENT_CELLS\tifs\agus_maria", r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\FLUORESCENT_CELLS\tifs\bf_0_fluo_1")

three_to_two_channel_folder(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\FLUORESCENT_CELLS\tifs\3_channels", r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\FLUORESCENT_CELLS\tifs\bf_0_fluo_1")

correct_4_channel_folder(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\FLUORESCENT_CELLS\tifs\4_channels", r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\FLUORESCENT_CELLS\tifs\corrected_4_channels")

print("all selected tasks done")
